<a href="https://colab.research.google.com/github/Schiavi13/analisis-impacto-covid19-en-inversion-ecoeficiencia-empresarial-2019-2023/blob/luis_001/src/ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Instalación de dependencias
Se instala librería de conexión con MySQL

In [1]:
# Instalar driver de MySQL y SQLAlchemy
!pip install PyMySQL SQLAlchemy

In [72]:
# Upgrade de pandas 3.0
!pip install --upgrade pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.


# 2. Importación de librerías

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types
import pymysql

In [3]:
print(pd.__version__)

3.0.2


# 3. Extracción de credenciales de conexión con base de datos
Se extraen las credenciales de conexión con la base de datos MySQL y se realiza un insert de test

In [4]:
# Extraer credenciales
from google.colab import userdata

MYSQL_HOST = userdata.get('MYSQL_HOST')
MYSQL_PORT = int(userdata.get('MYSQL_PORT'))
MYSQL_USER = userdata.get('MYSQL_USER')
MYSQL_PASS = userdata.get('MYSQL_PASS')
MYSQL_DB = userdata.get('MYSQL_DB')

In [98]:
# Crear el string de conexión para SQLAlchemy
DATABASE_URL = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"

# Crear la conexión usando SQLAlchemy
engine = create_engine(DATABASE_URL)

# Establecer una conexión directa con pymysql para operaciones que lo requieran (como commits)
connection = pymysql.connect(host = MYSQL_HOST,
                             port = MYSQL_PORT,
                             user = MYSQL_USER,
                             password = MYSQL_PASS,
                             db = MYSQL_DB,
                             cursorclass = pymysql.cursors.DictCursor)

In [ ]:
# Inserta registro de pais
with connection:
    with connection.cursor() as cursor:
        # nuevo registro
        sql = "INSERT INTO `pais` (`nombre_pais`) VALUES (%s)"
        cursor.execute(sql, ('colombia'))

    # commit para guardar los cambios
    connection.commit()

    with connection.cursor() as cursor:
        # Leer el registro
        sql = "SELECT `id_pais`, `nombre_pais` FROM `pais`"
        cursor.execute(sql)
        result = cursor.fetchone()
        print(result)

{'id_pais': 1, 'nombre_pais': 'colombia'}


# 4. Extracción de datos

In [57]:
# Extrae casos covid
df_casos_covid = pd.read_csv("/content/drive/MyDrive/Bootcamp Análisis de Datos/Grupo Q/data/procesados/covid/colombia_casos_covid19_diarios.csv")
df_casos_covid.shape

(809, 3)

In [7]:
df_casos_covid.head()

,fecha,nuevos_casos,total_casos
0,6/03/2020,1,1
1,7/03/2020,0,1
2,8/03/2020,0,1
3,9/03/2020,2,3
4,10/03/2020,0,3


In [72]:
# Extrae datos encuesta
df_encuesta_anual_servicios = pd.read_csv("/content/drive/MyDrive/Bootcamp Análisis de Datos/Grupo Q/data/clean/datos_unificados.csv")
df_encuesta_anual_servicios.shape

(30488, 39)

In [73]:
df_encuesta_anual_servicios.head()

,anio,id_empresa,id_sector,nombre_sector,anio_inicio,id_pais,nombre_pais,total_ingresos,personal_ocupado_total,personal_remunerado,...,realiza_otras_inv_amb,realiza_gasto_red_emision_aire,realiza_gasto_evita_ruido_vibra,realiza_gasto_energia_limp,realiza_gasto_ahorro_agua,realiza_gasto_terceros_amb,realiza_gasto_inst_med_agua,realiza_gasto_protec_suelo,realiza_gasto_disp_desechos,realiza_otros_gastos_amb
0,2019,10001,I1,alojamiento,1992.0,1,Colombia,2690535.0,42.0,42.0,...,0,0,0,0,0,0,0,0,0,0
1,2019,10002,I1,alojamiento,1994.0,1,Colombia,5198956.0,75.0,74.0,...,0,0,0,0,0,0,0,0,0,0
2,2019,10003,I1,alojamiento,1999.0,1,Colombia,14153289.0,192.0,75.0,...,0,0,0,0,0,0,0,0,0,0
3,2019,10004,I1,alojamiento,2009.0,1,Colombia,10684133.0,39.0,39.0,...,0,0,0,0,0,0,0,0,0,0
4,2019,10005,I2,expendio de alimentos,1957.0,1,Colombia,26319862.0,227.0,50.0,...,0,1,0,0,1,1,0,0,0,0


In [74]:
df_encuesta_anual_servicios.info()

<class 'pandas.DataFrame'>
RangeIndex: 30488 entries, 0 to 30487
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   anio                             30488 non-null  int64  
 1   id_empresa                       30488 non-null  int64  
 2   id_sector                        30488 non-null  str    
 3   nombre_sector                    30488 non-null  str    
 4   anio_inicio                      30487 non-null  float64
 5   id_pais                          30488 non-null  int64  
 6   nombre_pais                      30488 non-null  str    
 7   total_ingresos                   30484 non-null  float64
 8   personal_ocupado_total           30487 non-null  float64
 9   personal_remunerado              30487 non-null  float64
 10  contrata_personal_amb            24377 non-null  float64
 11  personal_mujeres                 30487 non-null  float64
 12  personal_hombres             

In [75]:
# Muestra tipos de datos
print(df_encuesta_anual_servicios.dtypes)

anio                                 int64
id_empresa                           int64
id_sector                              str
nombre_sector                          str
anio_inicio                        float64
id_pais                              int64
nombre_pais                            str
total_ingresos                     float64
personal_ocupado_total             float64
personal_remunerado                float64
contrata_personal_amb              float64
personal_mujeres                   float64
personal_hombres                   float64
gastos_totales                     float64
gasto_gestion_amb                  float64
gasto_personal_total               float64
imp_ind_com                        float64
imp_otros                          float64
gasto_remuneracion_personal        float64
gasto_servicios_publicos           float64
gasto_energia_electrica            float64
gasto_gas_natural                  float64
realiza_inv_red_emision_aire         int64
realiza_inv

In [76]:
# Transforma tipos de datos de la encuesta anual de servicios
dict_encuesta_anual_servicios_dtypes = {
    'anio_inicio': 'Int64',
    'personal_ocupado_total': 'Int64',
    'personal_remunerado': 'Int64',
    'contrata_personal_amb': 'Int64',
    'personal_mujeres': 'Int64',
    'personal_hombres': 'Int64',
}
df_encuesta_anual_servicios = df_encuesta_anual_servicios.astype(dict_encuesta_anual_servicios_dtypes)
print(df_encuesta_anual_servicios.dtypes)

anio                                 int64
id_empresa                           int64
id_sector                              str
nombre_sector                          str
anio_inicio                          Int64
id_pais                              int64
nombre_pais                            str
total_ingresos                     float64
personal_ocupado_total               Int64
personal_remunerado                  Int64
contrata_personal_amb                Int64
personal_mujeres                     Int64
personal_hombres                     Int64
gastos_totales                     float64
gasto_gestion_amb                  float64
gasto_personal_total               float64
imp_ind_com                        float64
imp_otros                          float64
gasto_remuneracion_personal        float64
gasto_servicios_publicos           float64
gasto_energia_electrica            float64
gasto_gas_natural                  float64
realiza_inv_red_emision_aire         int64
realiza_inv

In [77]:
# Estandariza id_sector y nombre_sector para empresas con datos conflictivos
# Identifica empresas con más de un id_sector unico one único
id_empresa_conflictivas = df_encuesta_anual_servicios.groupby('id_empresa')['id_sector'].nunique()
id_empresa_conflictivas = id_empresa_conflictivas[id_empresa_conflictivas > 1].index.tolist()

if id_empresa_conflictivas:
    print(f"Estandarizando {len(id_empresa_conflictivas)} empresas con múltiples sectores.")

    mapeo_sector = {}
    for emp_id in id_empresa_conflictivas:
        temp_df = df_encuesta_anual_servicios[df_encuesta_anual_servicios['id_empresa'] == emp_id]

        # id_sector más frecuente
        mode_id_sector = temp_df['id_sector'].mode()[0]

        # nombre_sector correspondiente
        mode_nombre_sector = temp_df[temp_df['id_sector'] == mode_id_sector]['nombre_sector'].mode()[0]

        mapeo_sector[emp_id] = {'id_sector': mode_id_sector, 'nombre_sector': mode_nombre_sector}

    # aplica los cambios en el dataframe de encuesta
    for emp_id, sector_info in mapeo_sector.items():
        mask = df_encuesta_anual_servicios['id_empresa'] == emp_id
        df_encuesta_anual_servicios.loc[mask, 'id_sector'] = sector_info['id_sector']
        df_encuesta_anual_servicios.loc[mask, 'nombre_sector'] = sector_info['nombre_sector']

    print("id_sector y nombre_sector estandarizados para empresas confictivas.")
else:
    print("No hay empresas con conflictos.")


Estandarizando 100 empresas con múltiples sectores.
id_sector y nombre_sector estandarizados para empresas confictivas.


# 5. Creación de tablas normalizadas
Se crea un dataframe por cada tabla de la base de datos

In [58]:
# Agregar clave foranea del pais
df_casos_covid.insert(1, "id_pais", 1)
df_casos_covid.head()

,fecha,id_pais,nuevos_casos,total_casos
0,6/03/2020,1,1,1
1,7/03/2020,1,0,1
2,8/03/2020,1,0,1
3,9/03/2020,1,2,3
4,10/03/2020,1,0,3


In [59]:
# Tamaño del dataframe casos_covid
df_casos_covid.shape

(809, 4)

In [60]:
# Define tipos de datos de casos_covid
dict_casos_covid_dtypes = {
    'fecha': 'datetime64[ns]',
    'id_pais': int,
    'nuevos_casos': int,
    'total_casos': int
}

In [61]:
# Transforma tipos de datos de casos_covid
df_casos_covid = df_casos_covid.astype(dict_casos_covid_dtypes)
print(df_casos_covid.dtypes)

fecha           datetime64[ns]
id_pais                  int64
nuevos_casos             int64
total_casos              int64
dtype: object


In [62]:
# Crea dataframe para tabla empresa_opera_pais
df_empresa_opera_pais = df_encuesta_anual_servicios[["id_empresa", "id_pais"]].drop_duplicates()
df_empresa_opera_pais.head()

,id_empresa,id_pais
0,10001,1
1,10002,1
2,10003,1
3,10004,1
4,10005,1


In [63]:
# Tamaño del dataframe empresa_opera_pais
df_empresa_opera_pais.shape

(7261, 2)

In [78]:
# Crea dataframe para tabla sector_economico
df_sector_economico = df_encuesta_anual_servicios[["id_sector", "nombre_sector"]].drop_duplicates()
print(df_sector_economico.sort_values(by='id_sector'))

      id_sector                                      nombre_sector
8            H1                          auxiliares del transporte
18501        H1         transporte de carga maritimo y de cabotaje
110          H2                                 postales y correos
0            I1                                        alojamiento
4            I2                              expendio de alimentos
4296         J0                                            edicion
141          J1              produccion, distribucion y exhibicion
12165        J1                                    otros servicios
97           J2                 prog y trans de radio y television
19           J3                  actividades de telecomunicaciones
151          J4                       inform y serv de informacion
23           LN                         inmobiliarias y alquileres
25           M1             actividades profesionales, cientificas
115          M2                                         public

In [79]:
# Ajuste de id_sector duplicado
df_encuesta_anual_servicios['nombre_sector'] = df_encuesta_anual_servicios['nombre_sector'].mask(df_encuesta_anual_servicios['id_sector']=='H1',
                                                                                                 'auxiliares del transporte')
df_encuesta_anual_servicios['nombre_sector'] = df_encuesta_anual_servicios['nombre_sector'].mask(df_encuesta_anual_servicios['id_sector']=='J1',
                                                                                                 'produccion, distribucion y exhibicion')
df_encuesta_anual_servicios['nombre_sector'] = df_encuesta_anual_servicios['nombre_sector'].mask(df_encuesta_anual_servicios['id_sector']=='N4',
                                                                                                 'actividades administrativas')
df_encuesta_anual_servicios['nombre_sector'] = df_encuesta_anual_servicios['nombre_sector'].mask(df_encuesta_anual_servicios['id_sector']=='Q',
                                                                                                 'salud humana')
df_sector_economico = df_encuesta_anual_servicios[["id_sector", "nombre_sector"]].drop_duplicates()
print(df_sector_economico.sort_values(by='id_sector'))

     id_sector                                      nombre_sector
8           H1                          auxiliares del transporte
110         H2                                 postales y correos
0           I1                                        alojamiento
4           I2                              expendio de alimentos
4296        J0                                            edicion
141         J1              produccion, distribucion y exhibicion
97          J2                 prog y trans de radio y television
19          J3                  actividades de telecomunicaciones
151         J4                       inform y serv de informacion
23          LN                         inmobiliarias y alquileres
25          M1             actividades profesionales, cientificas
115         M2                                         publicidad
12          N2                                 agencias de viajes
24          N3                 obtencion y suministro de personal
58        

In [80]:
# Tamaño del dataframe sector_economico
df_sector_economico.shape

(20, 2)

In [81]:
# Crea dataframe para tabla empresa
df_empresa = df_encuesta_anual_servicios[["id_empresa", "id_sector", "anio_inicio"]].drop_duplicates()
df_empresa.head()

,id_empresa,id_sector,anio_inicio
0,10001,I1,1992
1,10002,I1,1994
2,10003,I1,1999
3,10004,I1,2009
4,10005,I2,1957


In [82]:
# Tamaño dataframe empresa
df_empresa.shape

(9281, 3)

In [83]:
# Asegurar que anio_inicio sea el mínimo para cada id_empresa, id_sector y nombre_sector
min_anio_inicio = df_encuesta_anual_servicios.groupby(['id_empresa', 'id_sector', 'nombre_sector'])['anio_inicio'].min().reset_index()

# Actualizar el anio_inicio en df_encuesta_anual_servicios
df_encuesta_anual_servicios = df_encuesta_anual_servicios.merge(
    min_anio_inicio,
    on=['id_empresa', 'id_sector', 'nombre_sector'],
    how='left',
    suffixes=('', '_min')
)
df_encuesta_anual_servicios['anio_inicio'] = df_encuesta_anual_servicios['anio_inicio_min']
df_encuesta_anual_servicios = df_encuesta_anual_servicios.drop(columns=['anio_inicio_min'])

# Recrear el dataframe df_empresa con los anio_inicio ajustados
df_empresa = df_encuesta_anual_servicios[["id_empresa", "id_sector", "anio_inicio"]].drop_duplicates()
df_empresa.head()

,id_empresa,id_sector,anio_inicio
0,10001,I1,1992
1,10002,I1,1994
2,10003,I1,1999
3,10004,I1,2009
4,10005,I2,1957


In [84]:
# Tamaño del dataframe empresa después del ajuste
df_empresa.shape

(7261, 3)

In [24]:
# Crea dataframe para tabla personal
df_personal = df_encuesta_anual_servicios[["anio", "id_empresa", "personal_ocupado_total", "personal_remunerado", "contrata_personal_amb",
                                           "personal_mujeres", "personal_hombres"]].drop_duplicates()
df_personal.head()

,anio,id_empresa,personal_ocupado_total,personal_remunerado,contrata_personal_amb,personal_mujeres,personal_hombres
0,2019,10001,42,42,1,26,16
1,2019,10002,75,74,0,39,36
2,2019,10003,192,75,0,72,120
3,2019,10004,39,39,1,24,15
4,2019,10005,227,50,1,81,146


In [25]:
# Tamaño dataframe personal
df_personal.shape

(30488, 7)

In [26]:
# Crea dataframe para tabla inversiones
df_inversiones = df_encuesta_anual_servicios[["anio", "id_empresa", "realiza_inv_red_emision_aire", "realiza_inv_evita_ruido_vibra",
                                              "realiza_inv_energia_limp", "realiza_inv_ahorro_agua", "realiza_inv_inst_med_agua",
                                              "realiza_inv_protec_suelo", "realiza_inv_disp_desechos", "realiza_otras_inv_amb"]].drop_duplicates()
df_inversiones.head()

,anio,id_empresa,realiza_inv_red_emision_aire,realiza_inv_evita_ruido_vibra,realiza_inv_energia_limp,realiza_inv_ahorro_agua,realiza_inv_inst_med_agua,realiza_inv_protec_suelo,realiza_inv_disp_desechos,realiza_otras_inv_amb
0,2019,10001,0,0,0,0,0,0,0,0
1,2019,10002,0,0,0,0,0,0,0,0
2,2019,10003,0,0,0,0,0,0,0,0
3,2019,10004,0,0,0,0,0,0,0,0
4,2019,10005,0,0,0,0,0,0,0,0


In [27]:
# Tamaño dataframe inversiones
df_inversiones.shape

(30488, 10)

In [28]:
# Crea dataframe para tabla ingresos
df_ingresos = df_encuesta_anual_servicios[["anio", "id_empresa", "total_ingresos"]].drop_duplicates()
df_ingresos.head()

,anio,id_empresa,total_ingresos
0,2019,10001,2690535.0
1,2019,10002,5198956.0
2,2019,10003,14153289.0
3,2019,10004,10684133.0
4,2019,10005,26319862.0


In [29]:
# Tamaño dataframe ingresos
df_ingresos.shape

(30488, 3)

In [30]:
# Crea dataframe para tabla gastos
df_gastos = df_encuesta_anual_servicios[["anio", "id_empresa", "gasto_gestion_amb", "gastos_totales", "imp_ind_com", "imp_otros",
                                         "gasto_personal_total", "gasto_remuneracion_personal", "gasto_servicios_publicos",
                                         "gasto_energia_electrica", "gasto_gas_natural", "realiza_gasto_red_emision_aire",
                                         "realiza_gasto_evita_ruido_vibra", "realiza_gasto_energia_limp", "realiza_gasto_ahorro_agua",
                                         "realiza_gasto_terceros_amb", "realiza_gasto_inst_med_agua", "realiza_gasto_protec_suelo",
                                         "realiza_gasto_disp_desechos", "realiza_otros_gastos_amb"]].drop_duplicates()
df_gastos.head()

,anio,id_empresa,gasto_gestion_amb,gastos_totales,imp_ind_com,imp_otros,gasto_personal_total,gasto_remuneracion_personal,gasto_servicios_publicos,gasto_energia_electrica,gasto_gas_natural,realiza_gasto_red_emision_aire,realiza_gasto_evita_ruido_vibra,realiza_gasto_energia_limp,realiza_gasto_ahorro_agua,realiza_gasto_terceros_amb,realiza_gasto_inst_med_agua,realiza_gasto_protec_suelo,realiza_gasto_disp_desechos,realiza_otros_gastos_amb
0,2019,10001,0.0,2287702.0,31545.0,14564.0,875670.0,806770.0,57579.0,244691.0,104802.0,0,0,0,0,0,0,0,0,0
1,2019,10002,0.0,5020785.0,3906.0,102267.0,1498992.0,1490082.0,170977.0,425162.0,61113.0,0,0,0,0,0,0,0,0,0
2,2019,10003,0.0,15309856.0,168172.0,102069.0,2365296.0,2199361.0,201072.0,1410618.0,263741.0,0,0,0,0,0,0,0,0,0
3,2019,10004,0.0,8978494.0,88735.0,0.0,695317.0,695317.0,345980.0,456870.0,0.0,0,0,0,0,0,0,0,0,0
4,2019,10005,227327.0,29103165.0,146999.0,358035.0,2886717.0,2498261.0,394582.0,1251610.0,121518.0,1,0,0,1,1,0,0,0,0


In [31]:
# Tamaño dataframe gastos
df_gastos.shape

(30488, 20)

# 6. Carga de datos a base de datos
Se cargan los dataframes a la base de datos normalizada de manera remota

In [32]:
# Carga tabla casos_covid
with engine.connect() as conn:
    df_casos_covid.to_sql('casos_covid', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

# Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `casos_covid`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 809}


In [36]:
df_sector_economico.head()

,id_sector,nombre_sector
0,I1,alojamiento
4,I2,expendio de alimentos
8,H1,auxiliares del transporte
12,N2,agencias de viajes
19,J3,actividades de telecomunicaciones


In [49]:
# Carga tabla sector_economico
with engine.connect() as conn:
    df_sector_economico.to_sql('sector_economico', conn, if_exists = 'delete_rows', index = False,)
    conn.commit()

# Verifica l carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `sector_economico`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 20}


In [87]:
# Carga tabla empresa
with engine.connect() as conn:
    df_empresa.to_sql('empresa', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

# Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `empresa`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 7261}


In [90]:
# Carga tabla empresa_opera_pais
with engine.connect() as conn:
    df_empresa_opera_pais.to_sql('empresa_opera_pais', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

#Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `empresa_opera_pais`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 7261}


In [92]:
# Carga tabla ingresos
with engine.connect() as conn:
    df_ingresos.to_sql('ingresos', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

# Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `ingresos`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 30488}


In [94]:
# @title
# Carga la tabla personal
with engine.connect() as conn:
    df_personal.to_sql('personal', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

#Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `personal`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 30488}


In [97]:
# Carga tabla inversiones
with engine.connect() as conn:
    df_inversiones.to_sql('inversiones', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

# Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `inversiones`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 30488}


In [99]:
# Carga tabla gastos
with engine.connect() as conn:
    df_gastos.to_sql('gastos', conn, if_exists = 'delete_rows', index = False)
    conn.commit()

# Verifica la carga de datos
with connection.cursor() as cursor:
    sql = "SELECT COUNT(*) FROM `gastos`"
    cursor.execute(sql)
    result = cursor.fetchone()
    print(result)

{'COUNT(*)': 30488}


In [100]:
connection.close()